# Molecule Explorer — Drug Discovery & Cheminformatics

**Project 01: Understanding Molecules with Python + RDKit**

This notebook is designed as a **YouTube teaching project**.  
The goal is to start with real chemical data and progressively move from:

**SMILES → Molecular Structure → Atoms & Bonds → Molecular Properties → Descriptors → Fingerprints → Statistical EDA → Interactive Molecule Explorer**

### Learning objectives

By the end of this project, you should be able to:

1. Load and inspect the ESOL/Delaney solubility dataset.
2. Understand what a SMILES string represents.
3. Convert SMILES into RDKit molecular objects.
4. Validate molecular structures.
5. Visualize 2D molecular structures.
6. Inspect atoms and bonds programmatically.
7. Calculate common molecular properties/descriptors.
8. Generate molecular fingerprints.
9. Explore descriptor distributions and correlations.
10. Build a simple interactive Molecule Explorer.

> **Important:** The PyCaret/QSAR modeling experiment is intentionally kept out of this notebook. It belongs to the next stage of the roadmap: **ESOL Solubility Prediction / QSAR**. This notebook focuses on *understanding and exploring molecules*.


## 1. Project Roadmap

```text
                 MOLECULE EXPLORER
                         │
            ┌────────────┴────────────┐
            │                         │
       ESOL Dataset             Molecule Input
            │                         │
            ▼                         ▼
       Inspect Data              SMILES
            │                         │
            ▼                         ▼
       Validate Data          RDKit Mol Object
                                      │
                  ┌───────────────────┼───────────────────┐
                  ▼                   ▼                   ▼
                Atoms               Bonds            Structure
                                      │
                                      ▼
                              Molecular Properties
                                      │
                     ┌────────────────┼────────────────┐
                     ▼                ▼                ▼
                    MW              LogP              TPSA
                     ▼                ▼                ▼
                    HBD              HBA             Rings
                                      │
                                      ▼
                              Fingerprint
                                      │
                                      ▼
                                EDA / Analysis
                                      │
                                      ▼
                         Interactive Molecule Explorer
```


## 2. Environment Setup

We use:

- **Pandas / NumPy** — data handling
- **Matplotlib / Seaborn** — visualization
- **RDKit** — cheminformatics and molecular structures
- **ipywidgets** — interactive notebook controls

**ResearchPy is not used** because the original notebook encountered a compatibility error in the current Python environment.


In [1]:
# Install only if these packages are not already installed
%pip install pandas numpy matplotlib seaborn rdkit ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Core imports
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# RDKit
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw, rdMolDescriptors, rdFingerprintGenerator

# Interactive widgets
import ipywidgets as widgets
from IPython.display import display, clear_output


In [3]:
print("Pandas:", pd.__version__)
print("RDKit:", Chem.rdBase.rdkitVersion)


Pandas: 3.0.5
RDKit: 2026.03.6


## 3. Load the ESOL / Delaney Dataset

The dataset contains **1,128 molecules** and includes SMILES strings, molecular-property columns, and measured solubility values.

The original dataset columns include:

- `Compound ID`
- `ESOL predicted log solubility in mols per litre`
- `Minimum Degree`
- `Molecular Weight`
- `Number of H-Bond Donors`
- `Number of Rings`
- `Number of Rotatable Bonds`
- `Polar Surface Area`
- `measured log solubility in mols per litre`
- `smiles`


In [6]:
df = pd.read_csv("delaney-processed.csv")

print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])


FileNotFoundError: [Errno 2] No such file or directory: 'delaney-processed.csv'

In [ ]:
df.head()


In [ ]:
df.columns.tolist()


### First observation

Before doing cheminformatics, we first understand the **data structure**.

A useful habit in drug-discovery projects is:

> **Never start modeling before understanding the dataset.**


In [ ]:
df.info()


In [ ]:
print("Dataset shape:", df.shape)
print("\nMissing values:")
display(df.isnull().sum())


## 4. Convert SMILES → RDKit Molecules

A SMILES string is a text representation of a chemical structure.

For example:

```text
c1ccccc1
```

represents benzene.

RDKit allows us to convert that text representation into a molecular graph that Python can inspect and manipulate.


In [ ]:
df["mol"] = df["smiles"].apply(Chem.MolFromSmiles)

df[["Compound ID", "smiles", "mol"]].head()


### What is an RDKit `Mol` object?

The `Mol` object is RDKit's representation of a molecule.

It contains information about:

- atoms
- bonds
- connectivity
- aromaticity
- stereochemistry
- molecular structure

This is the bridge between **chemical notation** and **computational chemistry**.


## 5. Validate the Molecular Structures

Not every SMILES string is guaranteed to parse successfully.

We check for invalid structures before calculating descriptors.


In [ ]:
invalid_count = df["mol"].isna().sum()
valid_count = df["mol"].notna().sum()

print("Total molecules:", len(df))
print("Valid molecules:", valid_count)
print("Invalid molecules:", invalid_count)


## 6. Select a Molecule for Exploration

For the following examples, we will inspect one molecule from the dataset.

You can change `idx` to explore another molecule.


In [ ]:
idx = 10

mol = df.loc[idx, "mol"]
compound_id = df.loc[idx, "Compound ID"]
smiles = df.loc[idx, "smiles"]

print("Compound:", compound_id)
print("SMILES:", smiles)


## 7. Visualize the Molecular Structure

This is where cheminformatics becomes visually intuitive.

Instead of looking at a SMILES string, we can render the molecular graph as a 2D structure.


In [ ]:
Draw.MolToImage(mol, size=(500, 400))


## 8. Explore Atoms

A molecule can be represented as a graph:

- **Nodes = atoms**
- **Edges = bonds**

RDKit lets us inspect every atom individually.


In [ ]:
for atom in mol.GetAtoms():
    print(
        "Index:", atom.GetIdx(),
        "| Element:", atom.GetSymbol(),
        "| Atomic Number:", atom.GetAtomicNum()
    )


In [ ]:
print("Total atoms:", mol.GetNumAtoms())
print("Heavy atoms:", mol.GetNumHeavyAtoms())


### Why atom-level information matters

Atom-level information becomes important later for:

- molecular descriptors
- fingerprints
- graph neural networks
- molecular property prediction
- molecular generation

This graph representation is one of the foundations of modern cheminformatics.


## 9. Explore Bonds

Bonds connect the atoms in the molecular graph.

We can inspect:

- beginning atom
- ending atom
- bond type


In [ ]:
for bond in mol.GetBonds():
    print(
        "Begin:", bond.GetBeginAtomIdx(),
        "| End:", bond.GetEndAtomIdx(),
        "| Type:", bond.GetBondType()
    )


In [ ]:
print("Total bonds:", mol.GetNumBonds())


## 10. Calculate Basic Molecular Properties

Now we translate molecular structure into numerical features.

We will calculate:

| Property | Meaning |
|---|---|
| Molecular Weight (MW) | Approximate mass of the molecule |
| LogP | Lipophilicity-related descriptor |
| HBD | Hydrogen-bond donors |
| HBA | Hydrogen-bond acceptors |
| TPSA | Topological polar surface area |
| Rotatable Bonds | Approximate molecular flexibility |
| Ring Count | Number of rings |

These descriptors are widely used in cheminformatics and drug-discovery workflows.


In [ ]:
print("Molecular Weight:", Descriptors.MolWt(mol))
print("LogP:", Descriptors.MolLogP(mol))
print("H-Bond Donors:", Descriptors.NumHDonors(mol))
print("H-Bond Acceptors:", Descriptors.NumHAcceptors(mol))
print("TPSA:", Descriptors.TPSA(mol))
print("Rotatable Bonds:", Descriptors.NumRotatableBonds(mol))
print("Ring Count:", rdMolDescriptors.CalcNumRings(mol))


## 11. Calculate Descriptors for All 1,128 Molecules

A single molecule is useful for learning, but cheminformatics becomes powerful when we calculate descriptors for an entire chemical dataset.


In [ ]:
df["MW"] = df["mol"].apply(Descriptors.MolWt)
df["LogP"] = df["mol"].apply(Descriptors.MolLogP)
df["HBD"] = df["mol"].apply(Descriptors.NumHDonors)
df["HBA"] = df["mol"].apply(Descriptors.NumHAcceptors)
df["TPSA"] = df["mol"].apply(Descriptors.TPSA)
df["RotatableBonds"] = df["mol"].apply(Descriptors.NumRotatableBonds)
df["RingCount"] = df["mol"].apply(rdMolDescriptors.CalcNumRings)
df["Formula"] = df["mol"].apply(rdMolDescriptors.CalcMolFormula)


In [ ]:
descriptor_cols = [
    "MW",
    "LogP",
    "HBD",
    "HBA",
    "TPSA",
    "RotatableBonds",
    "RingCount"
]

molecule_summary = df[
    ["Compound ID", "smiles", "Formula"] + descriptor_cols
].copy()

molecule_summary.head()


## 12. Statistical Summary of Molecular Descriptors

Before plotting, we inspect:

- mean
- standard deviation
- minimum
- quartiles
- maximum

This gives us a first quantitative view of the chemical space.


In [ ]:
descriptor_summary = (
    molecule_summary[descriptor_cols]
    .describe()
    .T
    .round(2)
)

descriptor_summary


## 13. Univariate Analysis — Descriptor Distributions

We now ask:

> **How are molecular properties distributed across the dataset?**

This is useful for understanding the range and shape of the chemical space.


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 11))
axes = axes.flatten()

for i, col in enumerate(descriptor_cols):
    axes[i].hist(df[col], bins=30, edgecolor="black")
    axes[i].set_title(f"{col} Distribution")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Number of Molecules")

for j in range(len(descriptor_cols), len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()


## 14. Box Plots — Looking for Spread and Potential Outliers

Box plots help us see:

- median
- interquartile range
- overall spread
- potential extreme observations

An extreme value is not automatically an error. In chemical datasets, unusual molecules can be scientifically meaningful.


In [ ]:
plt.figure(figsize=(14, 7))
sns.boxplot(data=df[descriptor_cols])
plt.xticks(rotation=45)
plt.title("Molecular Descriptor Distributions")
plt.tight_layout()
plt.show()


## 15. Skewness Analysis

Skewness describes asymmetry in a distribution.

A positive value generally indicates a longer right tail.

For this dataset, several descriptors are strongly right-skewed, while LogP is much closer to symmetric.


In [ ]:
skewness = (
    df[descriptor_cols]
    .skew()
    .sort_values(ascending=False)
    .to_frame("Skewness")
)

skewness.round(3)


## 16. Multivariate Analysis — Correlation

Correlation helps us examine linear relationships between molecular descriptors.

For example, in this dataset, **HBA and TPSA show a strong positive correlation**.

Remember:

> Correlation describes association; it does not prove causation.


In [ ]:
corr = df[descriptor_cols].corr(method="pearson")

plt.figure(figsize=(10, 8))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)

plt.title("Correlation Between Molecular Descriptors")
plt.tight_layout()
plt.show()


## 17. Molecular Weight vs. Solubility

The ESOL dataset contains a measured solubility column.

We can use a simple scatter plot to begin asking chemical questions about the dataset.

The target column is:

`measured log solubility in mols per litre`


In [ ]:
target_col = "measured log solubility in mols per litre"

plt.figure(figsize=(8, 6))
plt.scatter(df["MW"], df[target_col], alpha=0.6)

plt.xlabel("Molecular Weight")
plt.ylabel("Measured logS")
plt.title("Molecular Weight vs. Measured Solubility")

plt.tight_layout()
plt.show()


## 18. Generate a Molecular Fingerprint

Descriptors summarize molecules using individual numerical properties.

Fingerprints take a different approach: they encode structural patterns as a vector of bits.

Here we use an RDKit Morgan fingerprint with:

- radius = 2
- fingerprint size = 2048 bits

This representation will become important in the next projects involving:

- molecular similarity
- chemical search
- clustering
- virtual screening


In [ ]:
generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048
)

fp = generator.GetFingerprint(mol)

print("Fingerprint type:", type(fp))
print("Fingerprint size:", fp.GetNumBits())
print("Number of on bits:", fp.GetNumOnBits())


## 19. Build a Clean Molecule Summary Table

This table is the main dataset we can use for exploration.

It combines:

**Compound identity + SMILES + Formula + molecular descriptors**


In [ ]:
molecule_summary = df[
    [
        "Compound ID",
        "smiles",
        "Formula",
        "MW",
        "LogP",
        "HBD",
        "HBA",
        "TPSA",
        "RotatableBonds",
        "RingCount"
    ]
].copy()

molecule_summary.head(10)


## 20. Find the Largest Molecules

A simple example of chemical-data exploration is sorting molecules by molecular weight.


In [ ]:
molecule_summary[
    ["Compound ID", "smiles", "MW", "LogP"]
].sort_values("MW", ascending=False).head(10)


## 21. Build an Interactive Molecule Explorer

Now we combine the ideas from the entire notebook.

The user can select a compound and see:

- molecular structure
- SMILES
- molecular formula
- MW
- LogP
- HBD
- HBA
- TPSA
- rotatable bonds
- ring count


In [ ]:
molecule_dropdown = widgets.Dropdown(
    options=list(molecule_summary["Compound ID"]),
    description="Molecule:",
    layout=widgets.Layout(width="450px")
)

display(molecule_dropdown)


In [ ]:
output = widgets.Output()

def explore_molecule(compound_id):
    row = df[df["Compound ID"] == compound_id].iloc[0]
    mol = row["mol"]

    with output:
        clear_output(wait=True)

        display(Draw.MolToImage(mol, size=(500, 400)))

        print("Compound ID:", row["Compound ID"])
        print("SMILES:", row["smiles"])
        print("Formula:", row["Formula"])
        print("Molecular Weight:", round(row["MW"], 2))
        print("LogP:", round(row["LogP"], 2))
        print("HBD:", row["HBD"])
        print("HBA:", row["HBA"])
        print("TPSA:", round(row["TPSA"], 2))
        print("Rotatable Bonds:", row["RotatableBonds"])
        print("Ring Count:", row["RingCount"])


In [ ]:
def on_molecule_change(change):
    if change["name"] == "value" and change["type"] == "change":
        explore_molecule(change["new"])

molecule_dropdown.observe(on_molecule_change)

display(output)

# Show the first molecule immediately
explore_molecule(molecule_dropdown.value)


# 🎯 Project Summary

We started with a table of chemical data and progressively converted it into a computational representation of molecules.

### What we learned

**1. Dataset**
- Loaded the ESOL / Delaney dataset.
- Inspected rows, columns, types, and missing values.

**2. Molecular representation**
- Converted SMILES strings into RDKit `Mol` objects.
- Validated molecular structures.

**3. Molecular graph**
- Explored atoms and bonds.
- Viewed molecules as graphs.

**4. Molecular properties**
- Calculated MW, LogP, HBD, HBA, TPSA, rotatable bonds, and ring count.

**5. Chemical data analysis**
- Calculated summary statistics.
- Examined distributions, skewness, outliers, and correlations.

**6. Molecular representation for AI**
- Generated a Morgan fingerprint.

**7. Application**
- Built an interactive Molecule Explorer.

### The bigger picture

```text
Chemical Structure
       ↓
      SMILES
       ↓
    RDKit Mol
       ↓
 ┌─────┴─────────┐
 ↓               ↓
Descriptors    Fingerprints
 ↓               ↓
EDA          Similarity/Search
 ↓               ↓
QSAR         Clustering
 ↓               ↓
Prediction   Virtual Screening
       ↓
Drug Discovery
```

### Next project

**Project 02 — Molecular Descriptor Calculator**

Then we can move toward:

**Project 03 — Lipinski Rule of Five / Drug-Likeness**

**Project 04 — Molecular Fingerprints**

**Project 05 — Molecular Similarity Search**

**Project 06 — Chemical Clustering**

**Project 07 — Chemical Space Visualization**

**Project 08 — ESOL Solubility Prediction / QSAR**


# 🎥 YouTube Discussion Structure

A natural way to explain this project on YouTube is:

### Part 1 — What are we building?
Introduce Molecule Explorer and explain why computational representation of molecules matters.

### Part 2 — What is SMILES?
Show a few SMILES examples and explain how text represents chemical structure.

### Part 3 — Meet RDKit
Convert SMILES → `Mol` and visualize the molecule.

### Part 4 — Molecules as graphs
Explain atoms as nodes and bonds as edges.

### Part 5 — Molecular properties
Explain MW, LogP, HBD, HBA, TPSA, rotatable bonds, and rings.

### Part 6 — From one molecule to 1,128 molecules
Calculate descriptors for the whole ESOL dataset.

### Part 7 — Chemical data analysis
Use histograms, box plots, skewness, and correlation.

### Part 8 — Fingerprints
Explain why a molecule can be represented as a binary structural fingerprint.

### Part 9 — Build the Explorer
Create the interactive dropdown-based molecule viewer.

### Part 10 — What's next?
Connect Molecule Explorer to similarity search, clustering, QSAR, docking, and AI-based drug discovery.

**Core message for the video:**

> We are not just learning Python. We are learning how a chemical structure becomes data that a computer can analyze.
